# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aijaz-khalique/flyrank-machine-learning-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I use a simple decision-support rule to rank content items that appear to have unusually weak CTR for their search-position context.

The main signal is `ctr_gap`: the difference between an item's observed CTR and the typical CTR for its position context. A more negative value means the item is receiving less CTR than expected relative to similar-position content.

I also use impression volume as a confidence signal. A negative CTR gap based on very few impressions may be noisy, while the same gap with more impressions is more useful for prioritization.

My score is:

`baseline_score = CTR gap severity + volume confidence`

Items with a large negative CTR gap and sufficient impression volume receive the highest score.

The rule produces one reason code:

* `LOW_CTR_FOR_POSITION` — the item has a negative CTR gap and enough impressions for the signal to be useful.

The action label is:

* `REVIEW_CTR_OPPORTUNITY` — inspect the title, snippet, search intent, and page presentation before making changes.

This is a directional decision-support rule, not a claim that the item will definitely decline or that changing it will improve Google rankings.


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

from datasets import load_dataset

# Paste your NEW Hugging Face token here
HF_TOKEN = "hf_PASTE_YOUR_NEW_TOKEN_HERE"

# Load the FlyRank dataset
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    split="train",
    token=HF_TOKEN
)

# Convert to DataFrame
df = dataset.to_pandas()

print("Data loaded successfully!\n")
print("Columns:")
print(df.columns.tolist())

print("\nRows:", len(df))

display(df.head())

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I rank items using two signals available at scoring time: CTR gap severity and impression volume.

First, I convert a more negative `ctr_gap` into a higher severity score. Then I add a smaller confidence component based on impression volume. The score is intentionally simple and interpretable.

I do not use product-generated flags, action labels, future-window outcomes, `trend_direction`, or any label-derived column as scoring inputs.


In [ ]:
# ---------------------------------------------------------
# Safety check: keep only scoring-time signals.
# ---------------------------------------------------------

required_columns = ["ctr_gap"]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Work on a copy.
queue = df.copy()

# ---------------------------------------------------------
# Find an impression column available in the feature frame.
# ---------------------------------------------------------

possible_impression_cols = [
    "impressions_28d",
    "impressions",
    "impressions_90d",
    "feature_impressions",
    "impressions_feature_window"
]

impression_col = next(
    (col for col in possible_impression_cols if col in queue.columns),
    None
)

if impression_col is None:
    raise ValueError(
        "No expected impression column was found. "
        f"Available columns are:\n{queue.columns.tolist()}"
    )

print("Using impression column:", impression_col)

# ---------------------------------------------------------
# 1. CTR gap severity
#
# More negative ctr_gap = higher priority.
# Positive gaps receive zero severity.
# ---------------------------------------------------------

queue["ctr_gap_severity"] = (
    -queue["ctr_gap"].clip(upper=0)
)

# Normalize to a 0-100 scale.
max_severity = queue["ctr_gap_severity"].max()

if max_severity > 0:
    queue["ctr_gap_score"] = (
        100 * queue["ctr_gap_severity"] / max_severity
    )
else:
    queue["ctr_gap_score"] = 0.0

# ---------------------------------------------------------
# 2. Volume confidence
#
# Log scaling prevents very large pages from dominating.
# ---------------------------------------------------------

queue["volume_raw"] = np.log1p(
    queue[impression_col].clip(lower=0)
)

max_volume = queue["volume_raw"].max()

if max_volume > 0:
    queue["volume_score"] = (
        100 * queue["volume_raw"] / max_volume
    )
else:
    queue["volume_score"] = 0.0

# ---------------------------------------------------------
# Final baseline score
#
# CTR gap is the main signal.
# Volume acts as a smaller confidence component.
# ---------------------------------------------------------

queue["baseline_score"] = (
    0.80 * queue["ctr_gap_score"]
    + 0.20 * queue["volume_score"]
)

# ---------------------------------------------------------
# One reason code.
# ---------------------------------------------------------

queue["reason_code"] = np.where(
    (queue["ctr_gap"] < 0) &
    (queue[impression_col] > 0),
    "LOW_CTR_FOR_POSITION",
    "NO_CTR_OPPORTUNITY"
)

# ---------------------------------------------------------
# One action label.
# ---------------------------------------------------------

queue["action_label"] = np.where(
    queue["reason_code"] == "LOW_CTR_FOR_POSITION",
    "REVIEW_CTR_OPPORTUNITY",
    "MONITOR"
)

# Rank highest-priority items first.
queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ---------------------------------------------------------
# Keep safe output columns.
# Do not export client names, URLs, or private queries.
# ---------------------------------------------------------

id_columns = [
    col for col in [
        "content_hash_id",
        "content_id",
        "client_hash_id"
    ]
    if col in queue.columns
]

output_columns = (
    ["rank"]
    + id_columns
    + [
        impression_col,
        "ctr_gap",
        "ctr_gap_score",
        "volume_score",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
)

output_columns = [
    col for col in output_columns
    if col in queue.columns
]

baseline_queue = queue[output_columns].copy()

# Write the required CSV.
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print(f"Rows written: {len(baseline_queue)}")

display(baseline_queue.head(20))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 items are ranked as decision-support candidates, not guaranteed problems. Each item has the same action because the rule is intentionally simple: a sufficiently negative CTR gap is treated as an opportunity to review whether the page is underperforming for its search-position context.

| Rank | Action                 | Reason code          | Confidence note                                                                     | What would make it wrong                                                                      |
| ---- | ---------------------- | -------------------- | ----------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------- |
| 1    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | Highest combined CTR-gap severity and volume confidence in this run.                | The position context may still be too broad, or the query mix may naturally have a lower CTR. |
| 2    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | Very strong negative CTR signal with useful volume.                                 | A recent title or snippet change may make the historical signal stale.                        |
| 3    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | High priority because the negative CTR gap is large and supported by impressions.   | The page may intentionally target a narrow audience with lower expected clicks.               |
| 4    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | Strong directional signal rather than a certainty.                                  | Search features or SERP layout may explain the lower CTR.                                     |
| 5    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | The rule observed below-context CTR and enough volume to justify review.            | The item may be competing with branded, answer-box, or other zero-click queries.              |
| 6    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | High score mainly reflects CTR underperformance.                                    | Position averages may hide important query-level differences.                                 |
| 7    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | The signal is sufficiently strong for manual inspection.                            | The measured period may be affected by seasonality or a temporary event.                      |
| 8    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | Negative CTR gap plus volume makes this a reasonable review candidate.              | The CTR benchmark may not match this content's actual intent or audience.                     |
| 9    | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | The item ranks highly under the same transparent rule as the other picks.           | A small number of high-impression queries may distort the aggregate CTR gap.                  |
| 10   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | The observed signal is strong enough for prioritization, not proof of a defect.     | The page could already be performing normally for its specific query mix.                     |
| 11   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | High relative priority from the combined severity and volume score.                 | SERP changes outside the page owner's control could explain the result.                       |
| 12   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | Directionally below the expected CTR context.                                       | The signal could disappear when measured over another period.                                 |
| 13   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | Enough observed evidence to place it in the review queue.                           | The page may have an appropriate but intentionally low-click informational intent.            |
| 14   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | The score prioritizes severity over unsupported certainty.                          | Query composition may differ from the group used to calculate expected CTR.                   |
| 15   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | A meaningful negative gap supported by available volume.                            | The page's snippet or title may already have changed after the feature window.                |
| 16   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | A reasonable manual-review candidate under this baseline.                           | Random variation could contribute to the measured gap.                                        |
| 17   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | High enough combined score to enter the top-20 queue.                               | The benchmark may not capture device, country, or query-intent differences.                   |
| 18   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | The evidence is directional and should be reviewed before action.                   | The observed CTR may be normal for the specific search results surrounding it.                |
| 19   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | The item remains a useful candidate under the fixed rule.                           | A temporary traffic pattern could make the measurement unrepresentative.                      |
| 20   | REVIEW_CTR_OPPORTUNITY | LOW_CTR_FOR_POSITION | Lowest-ranked item in the top 20, so confidence is weaker than the first few picks. | A small difference in the signals could move it outside the top 20.                           |

These reviews describe the ranking logic and uncertainty. The actual IDs and measured values are shown in the code output above, while the review intentionally avoids exposing client names, URLs, or private queries.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks are items where the negative CTR gap is present but the evidence is less convincing because impression volume is low or the final score is only slightly above nearby items.

I treat these as weaker decision-support candidates because a small amount of new data could change their rank substantially. The baseline therefore prioritizes manual review rather than automatic changes.

For leakage, the score uses only `ctr_gap` and impression volume from the feature frame. It does not use product-generated flags, action labels, future-window outcomes, `trend_direction`, decline labels, or any column derived from the target period.

This check is important because using those fields would make the baseline look artificially strong without representing a real scoring-time decision.


In [ ]:
# ---------------------------------------------------------
# Weak picks
# ---------------------------------------------------------

# Look at the bottom of the positive-priority queue.
weak_picks = (
    baseline_queue
    .loc[
        baseline_queue["action_label"]
        == "REVIEW_CTR_OPPORTUNITY"
    ]
    .tail(10)
    .copy()
)

print("Weakest 10 positive-priority picks:")
display(weak_picks)


# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

# These names represent fields that must NOT be scoring inputs.
forbidden_keywords = [
    "trend_direction",
    "label",
    "target",
    "future",
    "march",
    "flag",
    "action"
]

scoring_inputs = [
    "ctr_gap",
    impression_col
]

print("Scoring inputs:", scoring_inputs)

violations = []

for feature in scoring_inputs:
    feature_lower = feature.lower()

    for keyword in forbidden_keywords:
        if keyword in feature_lower:
            violations.append(feature)

if violations:
    raise ValueError(
        f"Potential leakage or forbidden input detected: {violations}"
    )

print("PASS: scoring inputs contain no obvious future, label, or product-flag fields.")

# Explicit check that common outcome fields were not used.
not_used = [
    "trend_direction",
    "is_declining_label",
    "action_label",
    "reason_code"
]

print("\nExplicitly excluded from scoring:")
for col in not_used:
    print("-", col)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.